In [ ]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
import re 
import numpy as np
import pandas as pd
import datetime
from datetime import datetime
import pandas_datareader.data as web

In [ ]:
datafile = 'Oct18-Oct19.csv'
data = pd.read_csv(datafile,encoding="ISO-8859-1")
data1 = data.copy()

data1["tweet_influence"] = data1['retweets']*0.6+data1['favorites']*0.3+data1['replies']*0.1


In [ ]:
data1 = data1.sort_values(by=['tweet_influence'],ascending=False)


In [ ]:
inf_30 = data1[:30]
inf_30

In [ ]:
class TwitterData_Initialize():
    data = []
    processed_data = []
    wordlist = []

    data_model = None
    data_labels = None
    is_testing = False
    
    def initialize(self, is_testing_set=False, from_cached=None):
        if from_cached is not None:
            self.data_model = inf_30
            return

        self.is_testing = is_testing_set

        if not is_testing_set:
            self.data = inf_30
            #self.data = self.data[['date','text']]
        else:
            self.data = inf_30
            not_null_text = 1 ^ pd.isnull(self.data["text"])
            not_null_date = 1 ^ pd.isnull(self.data["date"])
            self.data = self.data.loc[not_null_date & not_null_text, :]

        self.processed_data = self.data
        self.wordlist = []
        self.data_model = None
        self.data_labels = None

In [ ]:
data = TwitterData_Initialize()
data.initialize()
data.processed_data.head(5)

In [ ]:
class TwitterCleanuper:
    def iterate(self):
        for cleanup_method in [self.remove_urls,
                               self.remove_usernames,
                               self.remove_na,
                               self.remove_special_chars,
                               self.remove_numbers]:
            yield cleanup_method

    @staticmethod
    def remove_by_regex(tweets, regexp):
        tweets.loc[:, "text"].replace(regexp, "", inplace=True)
        return tweets

    def remove_urls(self, tweets):
        return TwitterCleanuper.remove_by_regex(tweets, regex.compile(r"http.?://[^\s]+[\s]?"))

    def remove_na(self, tweets):
        return tweets[tweets["text"] != "Not Available"]

    def remove_special_chars(self, tweets):  # it unrolls the hashtags to normal words
        for remove in map(lambda r: regex.compile(regex.escape(r)), [",", ":", "\"", "=", "&", ";", "%", "$",
                                                                     "@", "%", "^", "*", "(", ")", "{", "}",
                                                                     "[", "]", "|", "/", "\\", ">", "<", "-",
                                                                     "!", "?", ".", "'",
                                                                     "--", "---", "#"]):
            tweets.loc[:, "text"].replace(remove, "", inplace=True)
        return tweets

    def remove_usernames(self, tweets):
        return TwitterCleanuper.remove_by_regex(tweets, regex.compile(r"@[^\s]+[\s]?"))

    def remove_numbers(self, tweets):
        return TwitterCleanuper.remove_by_regex(tweets, regex.compile(r"\s?[0-9]+\.?[0-9]*"))

In [ ]:
class TwitterData_Cleansing(TwitterData_Initialize):
    def __init__(self, previous):
        self.processed_data = previous.processed_data
        
    def cleanup(self, cleanuper):
        t = self.processed_data
        for cleanup_method in cleanuper.iterate():
            if not self.is_testing:
                t = cleanup_method(t)
            else:
                if cleanup_method.__name__ != "remove_na":
                    t = cleanup_method(t)

        self.processed_data = t

In [ ]:
data = TwitterData_Cleansing(data)
data.cleanup(TwitterCleanuper())
# data.processed_data.head(10)

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize 
stop_words = set(stopwords.words('english')) 
word_tokens = []
nltk.download('punkt')



In [ ]:
for index in data.processed_data.iloc[:,6]:
    word_tokens.append(word_tokenize(index))

print('done')



In [ ]:
filtered_sentence = [] 


for word in word_tokens:
    this_filtered_sentence = [w.lower() for w in word if not w.lower() in stop_words] 
    filtered_sentence.append(this_filtered_sentence)

In [ ]:

data.processed_data['filtered']=filtered_sentence

In [ ]:
data.processed_data

In [ ]:
sentiment= SentimentIntensityAnalyzer()

data.processed_data['polarity_value']="Default"
data.processed_data['neg']=0.0
data.processed_data['pos']=0.0
data.processed_data['neu']=0.0
data.processed_data['compound']=0.0
for index,row in data.processed_data.iterrows():
    senti = sentiment.polarity_scores(row['text'])
    data.processed_data.set_value(index,'polarity_value',senti)
    data.processed_data.set_value(index,'neg',senti['neg'])
    data.processed_data.set_value(index,'pos',senti['pos'])
    data.processed_data.set_value(index,'neu',senti['neu'])
    data.processed_data.set_value(index,'compound',senti['compound'])
data.processed_data[:10]

In [ ]:
data1['text'][200692]
data1['text'][90715]

# negative comment from influential twitter users

In [ ]:
data.processed_data[data.processed_data["compound"]<0]['text']

In [ ]:
print ("List index-value are : ") 
for index, value in zip(range(len(data.processed_data)), data.processed_data): 
    print (index, value) 

# hashtag frequency 


In [ ]:
###### hashtag

#data1[data1["username"].str.match('com')] 
Nike_account_data = data1[(data1["username"]=='Nike' )| (data1["username"] == 'NikeNYC' )| (data1["username"] =='NikeLA')]
official_hashtag = Nike_account_data[pd.notnull(Nike_account_data['hashtags'])]

#official_hashtag = official_hashtag['hashtags'].str.split(" ", n=1, expand=True)

#official_hashtag[0].append(official_hashtag[1])
#Nike_account_data[pd.notnull(Nike_account_data['hashtags'])]

official_hashtag.head(10)



In [ ]:
official_hashtag = official_hashtag['hashtags'].str.split(" ", n=1, expand=True)


In [ ]:
x = pd.DataFrame(official_hashtag[0].append(official_hashtag[1]).dropna())
x.columns = (['hashtag'])
x

In [ ]:
y = x['hashtag'].value_counts()
y

In [ ]:
from pylab import rcParams
rcParams['figure.figsize'] = 10, 8
hashcount = pd.DataFrame(y)
hashcount[hashcount['hashtag']>1]
ax = hashcount[hashcount['hashtag']>1].plot.barh()

#### KNN cluster    (not work)

In [ ]:
# Importing libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
import string
import re
import numpy as np
from collections import Counter

In [ ]:
train_data=data1[1:50]

In [ ]:
stop = set(stopwords.words('english'))
exclude = set(string.punctuation)
lemma = WordNetLemmatizer()
 
# Cleaning the text sentences so that punctuation marks, stop words &amp; digits are removed
def clean(doc):
    stop_free = " ".join([i for i in doc.lower().split() if i not in stop])
    punc_free = ''.join(ch for ch in stop_free if ch not in exclude)
    normalized = " ".join(lemma.lemmatize(word) for word in punc_free.split())
    processed = re.sub(r"\d+","",normalized)
    y = processed.split()
    return y

In [ ]:
train_clean_sentences = []
for line in train_data['text']:
    line = line.strip()
    cleaned = clean(line)
    cleaned = ' '.join(cleaned)
    train_clean_sentences.append(cleaned)
 
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(train_clean_sentences)

In [ ]:
# Creating true labels for 30 training sentences
y_train = np.zeros(49)
y_train[10:20] = 1
y_train[20:50] = 2

In [ ]:
# Clustering the document with KNN classifier
modelknn = KNeighborsClassifier(n_neighbors=5)
modelknn.fit(X,y_train)
 
# Clustering the training 30 sentences with K-means technique
modelkmeans = KMeans(n_clusters=3, init='k-means++', max_iter=200, n_init=100)
modelkmeans.fit(X)

In [ ]:
test_sentences = y
test_clean_sentence = []
for test in test_sentences:
    cleaned_test = clean(test)
    cleaned = ' '.join(cleaned_test)
    cleaned = re.sub(r"\d+","",cleaned)
    test_clean_sentence.append(cleaned)
 
Test = vectorizer.transform(test_clean_sentence)
 
true_test_labels = ['ship','sad','angry']
predicted_labels_knn = modelknn.predict(Test)
predicted_labels_kmeans = modelkmeans.predict(Test)
 
print ("\nBelow 3 sentences will be predicted against the learned nieghbourhood and learned clusters:\n1. ",\
test_sentences[0],"\n2. ",test_sentences[1],"\n3. ",test_sentences[2])
print ("\n-------------------------------PREDICTIONS BY KNN------------------------------------------")
print ("\n",test_sentences[0],":",true_test_labels[np.int(predicted_labels_knn[0])],\
"\n",test_sentences[1],":",true_test_labels[np.int(predicted_labels_knn[1])],\
"\n",test_sentences[2],":",true_test_labels[np.int(predicted_labels_knn[2])])
 
print ("\n-------------------------------PREDICTIONS BY K-Means--------------------------------------")
print ("\nIndex of happy cluster : ",Counter(modelkmeans.labels_[0:10]).most_common(1)[0][0])
print ("Index of fear cluster : ",Counter(modelkmeans.labels_[10:20]).most_common(1)[0][0])
print ("Index of angry cluster : ",Counter(modelkmeans.labels_[20:30]).most_common(1)[0][0])
print ("\n",test_sentences[0],":",predicted_labels_kmeans[0],\
"\n",test_sentences[1],":",predicted_labels_kmeans[1],\
"\n",test_sentences[2],":",predicted_labels_kmeans[2])

In [ ]:
x = pd.DataFrame([data1[60:70]['text']])
x

In [ ]:
y = x.values.tolist()[0]
y

In [ ]:
nltk.download('wordnet')

In [ ]:
data.processed_data
train_clean_sentences = []
for line in data.processed_data['filtered']:
    for x in line:
        train_clean_sentences.append(x)
train_clean_sentences 

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(train_clean_sentences)

In [ ]:
X

In [ ]:
y_train = np.zeros(424)
y_train[10:20] = 1
y_train[20:424] = 2


In [ ]:
#modelknn = KNeighborsClassifier(n_neighbors=5)
#modelknn.fit(X,y_train)

In [ ]:
#modelkmeans = KMeans(n_clusters=3, init='k-means++', max_iter=200, n_init=100)
#modelkmeans.fit(X)

# data preprocessing for pyspark

In [ ]:
data_mention=data1[:3000][['username','mentions']].dropna()

In [ ]:
data_mention['mentions'] = data_mention['mentions'].apply(lambda x: x.split(' '))
data_mention = data_mention.explode('mentions')

In [ ]:
data_mention

In [ ]:
#cdata_mention.to_csv(r'mention2.csv')

In [ ]:
dfp = pd.read_csv("mention2.csv")

In [ ]:
dfp['mentions']=dfp['mentions'].str.replace(r'@','')

In [ ]:
dfp.to_csv(r'metions4.csv')

In [ ]:
df_jordan_positive.to_csv(r'df_jordan_pos.csv')

In [ ]:
stop = stopwords.words('english')